# 05 · Event check

**What this notebook does.** It checks the **event-related potentials (ERPs)**, the average LFP around the press, the tone and the
release, in all final sessions. This answers two questions:
1. **Are the event times right?** The press can be aligned to the TTL pulse or to the synced behaviour time; a sharper ERP means a better alignment.
2. **Are there more bad channels?** A channel whose ERP is much larger or smaller than its neighbours' in all three events is a candidate bad channel.
   The ones confirmed here were added to `extra_bad_channels.csv`.

It also shows an example ERP and how much of it is in the delta band (relevant for the delta results in 07).

**Run order in the project:** 01 → 02 → 03 → 04 → **05** → 06 → 07. Needs `final_sessions.csv`, `trials_filtered.csv`,
`extra_bad_channels.csv`, and the QC table of 04 (`qc_channels_all.csv`, for step 2).

**Steps**

| Step | What it does | Output |
|---|---|---|
| 1 | ERPs of every session, original and CMR (only with `RUN = True`) | per session `ERP\erps.npz`, `erp_checks.csv`, `erp_sharpness.csv`; 2 summary tables |
| 2 | Summary: TTL vs behaviour alignment; channels odd in all 3 events | `erp_candidates.csv` |
| 3 | Candidate channels in detail | `bad_channel_candidates_check.csv` + figures |
| 4 | Add the confirmed channels to `extra_bad_channels.csv` (append only) | `extra_bad_channels.csv` |
| 5 | ERPs for the LOCAL reference + comparison | per session `ERP\*_local*`; 2 summary tables |
| 6 | Example ERP of one channel (mean ± SD) | figure |
| 7 | The same ERP and its delta part (1–4 Hz) | figure |

Removed from the old notebook: three test cells on a single session (step 1 does the same for all sessions).

**How to use it:** run the settings cell first; then any cell. Nothing is written unless `SAVE = True` (figures and tables) or `RUN = True`
(the per-session ERP files). Functions: `fx_events.py` (+ `functions.py`); colours and saving: `fx_common.py`.

**The three references** (made by `functions.load_session` when a session is loaded; the saved recording itself has no reference):
- **original**: the 1000 Hz recording as saved (no re-referencing);
- **CMR** (common median reference): every channel minus the median of all channels at each time point. Removes signal shared by the whole probe;
- **LOCAL**: every channel minus the median of the channels 30–200 µm away on the probe. Removes more local shared signal.

Before referencing, the channels removed in preprocessing are already missing and the session's channels from
`extra_bad_channels.csv` are removed, so bad channels can't leak into the reference.

## Settings

**Paths:** `CODE_DIR`, `DATA_ROOT` with `PROC` (per-session ERP files go into `PROC\<session>\amir_data\ERP`), `AMIR`, `TABLES`
(temporarily the old folder), `FIG_DIR` (figures of this notebook), `ERP_DIR` (slide figures of steps 6–7).

**Saving and loops:** `SAVE = False` writes nothing; `True` writes figures to `FIG_DIR` and tables to `TABLES`.
`RUN = False` only reports the finished sessions; `True` computes the missing ones; `OVERWRITE = True` redoes all.
To save one cell elsewhere, add `folder=r"...", force=True` to its save line.

**Data loaded here:** `final`, `XB` (extra bad channels), `TR` (all trials) and `K` (kept trials). Event times on the neural clock (01):
press = `press_neural_s` (TTL, else behaviour time), tone = `tone_neural_s` (correct and late trials; early trials have no tone),
release = `release_neural_s`.

**Bad-channel decisions (step 4):** `NEW_BAD` = channel: (rat, reason). **Candidates (step 3):** `CANDIDATES`.
**Example (steps 6–7):** `EX_SESSION`.

In [ ]:
# ══ SETTINGS — run first ══
import os, sys, importlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

# ── paths ──
CODE_DIR  = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"                  # folder with the notebooks + fx_*.py + functions.py
DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"             # project data on the server
PROC      = os.path.join(DATA_ROOT, "2_processed")                                 # 1000 Hz recordings (+ per-session QC / ERP files)
AMIR      = os.path.join(DATA_ROOT, "amir_data")                                   # your own data folder (all outputs)
TABLES    = os.path.join(CODE_DIR, "tables")                                       # TEMPORARY: old place; later os.path.join(AMIR, "tables")
FIG_DIR   = os.path.join(AMIR, "figures", "05_event_check")                        # figures of this notebook (if SAVE)
ERP_DIR   = os.path.join(AMIR, "ERP")                                              # slide figures of steps 6–7 (if SAVE)

# ── saving and loops ──
SAVE      = False                                                                  # True = write figures and tables
RUN       = False                                                                  # True = compute missing per-session ERP files
OVERWRITE = False                                                                  # True = recompute every session

# ── bad-channel decisions (2026-10-23) ──
CANDIDATES = ["AP381", "AP158", "AP313", "AP329"]                                  # step 3: channels checked in detail
NEW_BAD = {"AP381": ("135", "event check: same signal ~4 % stronger, ERP 7x after CMR in 22/29 sessions"),
           "AP158": ("134", "event check: weaker/different signal (r 0.81 with neighbours), ERP 4-6x after CMR"),
           "AP313": ("134", "QC + check: extra noise (r 0.84 with neighbours), power 3.7x after CMR")}   # step 4

# ── example ──
EX_SESSION = "20260617_r0000135_wt_1944"                                           # steps 6–7

# ── code + data ──
sys.path.insert(0, CODE_DIR)
import fx_common as fxc, functions as fn, fx_events as fe
for m_ in (fxc, fn, fe): importlib.reload(m_)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
fxc.check_paths(PROC=PROC, TABLES=TABLES)
fxc.set_saving(SAVE, fig_dir=FIG_DIR, table_dir=TABLES)
read  = lambda name: pd.read_csv(os.path.join(TABLES, name + ".csv"), dtype={"rat": str}, low_memory=False)
final = read("final_sessions"); XB = pd.read_csv(os.path.join(TABLES, "extra_bad_channels.csv"))
TR    = read("trials_filtered"); K = TR[TR["keep"]]
REFS  = ["original", "CMR"]
print(f"final sessions: {len(final)} | kept trials: {len(K)} | extra bad channel rows: {len(XB)}")

## Step 1 · ERPs of all final sessions (original and CMR)

**What it does, per session** (`functions.erp_session`), with the kept trials:
1. **ERPs**: the LFP from −1 to +1 s around each event, averaged over trials, for every channel; the mean before the event is removed per trial.
   Events: Press, Tone (correct + late trials), Release, and the press twice more on the **same** trials: aligned to the TTL pulse
   (`Press_TTL`) and to the synced behaviour time (`Press_MA`). Original and CMR.
2. **Alignment sharpness:** peak-to-peak of the press ERP (−0.1 to 0.3 s) per channel, TTL / behaviour time. A misaligned event smears the
   average, so the better alignment gives the larger (sharper) ERP; ratio > 1 = TTL sharper.
3. **ERP outliers** (CMR): ERP size (log10 of its SD over time) vs the neighbours on the same shank (±75 µm); flagged if > 2× or < 0.5×,
   for Press, Tone and Release separately.

**With `RUN = True`** it writes per session `ERP\erps.npz` (mean ERPs), `ERP\erp_checks.csv` (outliers per channel × event) and
`ERP\erp_sharpness.csv` (written last = session finished). About 0.3–0.5 min per session.

**Saves:** `erp_sharpness_all.csv`, `erp_outliers_all.csv` (all sessions collected).

In [ ]:
fe.run_event_check(final, K, PROC, XB, run=RUN, overwrite=OVERWRITE, err_log=os.path.join(TABLES, "erp_errors.csv"))
erp_sharp, erp_outl = fe.collect_erp(final, PROC)
if erp_sharp is not None: fxc.save_table(erp_sharp, "erp_sharpness_all"); fxc.save_table(erp_outl, "erp_outliers_all")

## Step 2 · Event check summary

**What it does.**
1. **TTL vs behaviour time:** per rat and reference, the median TTL / behaviour ratio over sessions, the number of sessions where the TTL is
   sharper, and a **Wilcoxon signed-rank test** of log10(ratio) against 0 (no difference). Why log: a ratio of 2 and of 0.5 are equally far from 1.
2. **Candidate bad channels:** channels flagged in **all three** events (Press, Tone, Release) of a session, CMR. One odd event can be chance;
   all three point to the channel.
3. **Repeat offenders:** the same readout channel flagged in all 3 events in several sessions of a rat.
4. Whether these channels are also a power outlier in 04 (same session).

**Expected (2026-10-19, test session):** TTL / behaviour ≈ 0.97–0.99 = no difference.

**Input:** the collected ERP tables and `qc_channels_all.csv` (04). **Saves:** `erp_candidates.csv`.

In [ ]:
erp_sharp, erp_outl = fe.collect_erp(final, PROC)
candidates = fe.event_summary(erp_sharp, erp_outl, read("qc_channels_all"), final)
fxc.save_table(candidates, "erp_candidates")

## Step 3 · Candidate channels in detail

**What it does**, for each channel in `CANDIDATES`, in the session where it was flagged most strongly (all-3-event list first, else the 04 QC):
- power and ERP size (Press, Tone, Release) as a factor of its neighbours' median (±75 µm), original and CMR;
- from 60 s of raw signal: its median **correlation with the neighbours** vs the neighbours' correlation with each other, and its RMS vs theirs.
  A healthy channel moves together with its neighbours (r ≈ 0.99); a broken one is weaker, noisier or different;
- figures: spectra and press ERP of the channel (red) vs neighbours (grey), and 4 s of raw traces.

The candidates are loaded **without** themselves in the extra-bad list (they were added afterwards, in step 4), otherwise they would be missing.

**Saves:** `bad_channel_candidates_check.csv`, figures `candidate_<channel>.png`, `candidate_<channel>_traces.png`.

In [ ]:
ap_of = XB["channel_id"].astype(str).str.extract(r"(AP\d+)")[0]
cand_check, cand_figs = fe.candidate_check(CANDIDATES, read("erp_candidates"), read("qc_channels_all"), final, PROC, XB[~ap_of.isin(CANDIDATES)])
fxc.save_table(cand_check, "bad_channel_candidates_check")
for ap, (f1, f2) in cand_figs.items(): fxc.save_fig(f1, f"candidate_{ap}", dpi=150); fxc.save_fig(f2, f"candidate_{ap}_traces", dpi=150)
plt.show()

## Step 4 · Add the new bad channels to `extra_bad_channels.csv` (append only)

**Decided 2026-10-23** from steps 2–3:
- **AP381** (rat 135): same signal as its neighbours but ~4 % stronger → after CMR its ERP is 7× too large, at 3 positions;
- **AP158** (rat 134): weaker and partly different signal (0.6× power, r = 0.81 with neighbours vs 0.99) → ERP 4–6× too large after CMR;
- **AP313** (rat 134): extra noise (r = 0.84 with neighbours vs 0.99), power 3.7× after CMR, ERP normal.

Each is added for **every usable session of that rat** (as the hardware channels AP326 / AP362 / AP63 / AP90 on 2026-10-20), unless it is
already listed or was removed in preprocessing. Kept (not added): AP329 (rat 136; normal in the original reference, odd only after CMR at one
position), AP23 (only 2 sessions).

**Expected now:** "nothing to add" (they were added on 2026-10-23). **With `SAVE = True`** a backup of the old file is written first.

In [ ]:
add, xb_new = fe.new_bad_channels(NEW_BAD, XB, read("preprocessing_log"), read("usable_sessions"))
if len(add):
    fxc.save_table(XB, "extra_bad_channels_backup"); fxc.save_table(xb_new, "extra_bad_channels")

## Step 5 · Event check for the LOCAL reference

**What it does.** The same as step 1 on the **LOCAL** reference: ERPs at Press, Press TTL / behaviour (same trials), Tone (correct + late),
Release; alignment sharpness; ERP outliers vs neighbours for Press, Tone, Release. (`functions.erp_session` checks CMR only, so its building
blocks are called directly.) With `RUN = True` it writes per session `ERP\erps_local.npz`, `erp_checks_local.csv`, `erp_sharpness_local.csv`.

**Summary:** TTL / behaviour ratio for original, CMR and LOCAL; channels flagged in all 3 events, CMR vs LOCAL.

**Saves:** `erp_sharpness_all_local.csv`, `erp_outliers_all_local.csv`.

In [ ]:
fe.run_event_check_local(final, K, PROC, XB, run=RUN, overwrite=OVERWRITE)
sharp_local, outl_local = fe.collect_erp(final, PROC, local=True)
if sharp_local is not None:
    fxc.save_table(sharp_local, "erp_sharpness_all_local"); fxc.save_table(outl_local, "erp_outliers_all_local")
    fe.event_local_summary(sharp_local, outl_local, *fe.collect_erp(final, PROC))

## Step 6 · Example ERP of one channel

**What it does.** In `EX_SESSION`, picks the channel with the clearest response to **all three** events: from the saved mean ERPs (original),
for each event the largest deflection in its response window (press 0–0.4 s, tone 0–0.3 s, release −0.1–0.3 s) divided by the SD of the
baseline (−0.5 to −0.1 s); the channel whose **weakest** of the three is the highest. Then it recomputes that channel's ERP from the
recording and shows the mean ± SD across trials (−0.5 to 0.5 s).

**Saves:** `ERP press tone release <session> <channel>.png` in `ERP_DIR`.

In [ ]:
fig, EX_CH, ex_rec, ex_events = fe.example_erp(PROC, XB, TR, EX_SESSION)
fxc.save_fig(fig, f"ERP press tone release {EX_SESSION} {EX_CH.split('#')[-1]}", folder=ERP_DIR); plt.show()

## Step 7 · The example ERP and its delta part

**What it does.** For the channel of step 6: the mean ERP from −1 to +1 s (longer than shown, so the filter has clean edges), and the same
ERP band-passed into **delta (1–4 Hz)** (3rd-order Butterworth, run forward and backward = no phase shift). It reports the peak-to-peak size
in 0–0.4 s after each event for the full ERP, delta, theta (4–8), alpha (8–13) and beta (15–30 Hz), and the % of the ERP that is delta.

**Why:** a large part of the event-locked response is slow (delta). Delta power and delta coupling around the events (07) can therefore be
driven by the ERP itself; that is why the delta results in 07 are also computed with the ERP removed.

**Saves:** `ERP and delta <session> <channel>.png` in `ERP_DIR`.

In [ ]:
fig, EX_CH, ex_rec, ex_events = fe.example_erp(PROC, XB, TR, EX_SESSION); plt.close(fig)       # same channel as step 6
fig = fe.erp_delta_part(ex_rec, EX_CH, ex_events, EX_SESSION)
fxc.save_fig(fig, f"ERP and delta {EX_SESSION} {EX_CH.split('#')[-1]}", folder=ERP_DIR); plt.show()